**pandas** is a Python library for data manipulation and analysis. It provides data structures and operations for working with numerical tables and time series.

In this notebook, you will learn how to load a dataset and explore, slice, clean, and manipulate it with pandas.

## 1.1 Installing and Importing Pandas

Run the cell below to install pandas if it isn't already installed in your environment. If it's already installed, this cell has no effect.

In [7]:
#!pip install pandas

Import pandas under the conventional alias `pd`.

In [8]:
import pandas as pd

## 1.2 Importing a Dataset

This notebook uses `titanic.csv`, a dataset describing the passengers of the Titanic (age, sex, ticket class, whether they survived, etc.).

Run the cell below and upload `titanic.csv` (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select titanic.csv

Load the uploaded CSV into a pandas DataFrame and print it.

In [ ]:
df = pd.read_csv('/content/data.csv')
print(df)

unrelevent column drop

In [ ]:
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')
df.head()

Missing value check ও mean imputation

In [ ]:
import numpy as np

print("Missing values before:\n", df.isnull().sum()[df.isnull().sum() > 0])
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
print("Missing values after:", df.isnull().sum().sum())
df.head()

Categorical variable encode

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])   # M=1, B=0
print(dict(zip(le.classes_, le.transform(le.classes_))))
df.head()

Feature and target separate

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
df.head()

Min-Max Scaling on feature. not on target

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(X_scaled.head())
df.head()

preprocessing

In [ ]:
df = pd.read_csv('data.csv')
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])  # M=1, B=0

X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
df_scaled = X_scaled.copy()
df_scaled['diagnosis'] = y.values

Violin Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

selected_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
                      'smoothness_mean', 'concavity_mean', 'concave points_mean',
                      'radius_worst', 'texture_worst', 'concavity_worst']

melted = df_scaled.melt(id_vars='diagnosis', value_vars=selected_features,
                         var_name='features', value_name='value')

plt.figure(figsize=(14, 6))
sns.violinplot(x='features', y='value', hue='diagnosis', data=melted, split=True,
               palette=['#2ca02c', '#ff7f0e'])
plt.xticks(rotation=45, ha='right')
plt.title('Violin Plot: Distribution and Density (Benign=0 vs Malignant=1)')
plt.tight_layout()
plt.show()

Correlation Heatmap

In [ ]:
corr_matrix = X.corr()  # Pearson correlation coefficient

plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, cmap='Reds', annot=False, square=True, cbar_kws={'shrink': 0.7})
plt.title('Correlation Heatmap of All Features')
plt.tight_layout()
plt.show()

Highly correlated pairs

In [ ]:
threshold = 0.9
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j],
                                     round(corr_matrix.iloc[i, j], 3)))

print(f"Total highly correlated pairs (|corr| > {threshold}): {len(high_corr_pairs)}")
for pair in high_corr_pairs:
    print(pair)

 Correlation-based elimination(eq,4)

In [ ]:
threshold_corr = 0.9
corr_matrix = X.corr()
to_drop = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold_corr:
            to_drop.add(corr_matrix.columns[j])
X_reduced = X_scaled.drop(columns=list(to_drop))
df.head()

 Chi-Square (Eq. 5)

In [ ]:
from sklearn.feature_selection import chi2

feature_names = X_scaled.columns
votes = pd.DataFrame({'Feature': feature_names})
votes['Chi_square'] = 0

chi_scores, p_values = chi2(X_scaled, y)
chi_series = pd.Series(chi_scores, index=feature_names)
top_chi = chi_series.sort_values(ascending=False).head(16).index.tolist()
votes.loc[votes['Feature'].isin(top_chi), 'Chi_square'] = 1
df.head()

 Random Forest importance (Eq. 6)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_scaled, y)
rf_importance = pd.Series(rf.feature_importances_, index=feature_names)
top_rf = rf_importance.sort_values(ascending=False).head(16).index.tolist()
votes.loc[votes['Feature'].isin(top_rf), 'RF'] = 1
df.head()

 Extra Trees

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_estimators=200, random_state=42)
et.fit(X_scaled, y)
et_importance = pd.Series(et.feature_importances_, index=feature_names)
top_et = et_importance.sort_values(ascending=False).head(16).index.tolist()
votes.loc[votes['Feature'].isin(top_et), 'ExtraTrees'] = 1
df.head()

5. RFE

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe = RFE(LogisticRegression(max_iter=5000), n_features_to_select=16)
rfe.fit(X_scaled, y)
top_rfe = X_scaled.columns[rfe.support_].tolist()
votes.loc[votes['Feature'].isin(top_rfe), 'RFE'] = 1
df.head()

 7. Lasso (L1) (Eq. 7)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV
# LogisticRegression was imported in a previous cell, assuming it's available

cv = StratifiedKFold(5)
rfecv = RFECV(LogisticRegression(max_iter=5000), step=1, cv=cv, min_features_to_select=10)
rfecv.fit(X_scaled, y)
top_rfecv = X_scaled.columns[rfecv.support_].tolist()
votes.loc[votes['Feature'].isin(top_rfecv), 'RFECV'] = 1
df.head()

Final voting score (Table I)

In [ ]:
from sklearn.linear_model import LogisticRegression

lasso_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=5000)
lasso_model.fit(X_scaled, y)

lasso_features = X_scaled.columns[lasso_model.coef_[0] != 0].tolist()
votes['L1'] = 0
votes.loc[votes['Feature'].isin(lasso_features), 'L1'] = 1

votes['Final_R_score'] = votes[['Chi_square', 'RF', 'ExtraTrees', 'RFE', 'RFECV', 'L1']].sum(axis=1)
votes = votes.sort_values('Final_R_score', ascending=False).reset_index(drop=True)
print(votes.to_string(index=False))

Final 16 features

In [ ]:
final_16 = votes.head(16)['Feature'].tolist()
print("Final 16 selected features:", final_16)


Fig. 4 style plot

In [ ]:
plt.figure(figsize=(10, 8))
rf_importance.sort_values(ascending=False).head(16).plot(kind='barh', color='red')
plt.gca().invert_yaxis()
plt.title('Selection of the Most Important Features (RF Importance)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

D

In [ ]:
jhklytf

separate target and feature

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
df.head()

target variable encode

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)
df.head()

Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
df.head()

Base Model using as Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
df.head()